# Challenge Four: Programming an Agent Workflow

This notebook builds **Cloud Security Advisor**, an ADK `SequentialAgent` workflow that answers a
question, then verifies and refines the answer before returning it. The workflow runs four
sub-agents in a strict order:

1. **Greeter Agent** - Acknowledges the user's question and hands off to the workflow.
2. **Search Agent** - Researches and drafts the initial answer using the built-in `google_search` tool.
3. **Critique Agent** - Reviews the initial answer for accuracy, completeness, clarity, and missing security considerations.
4. **Refine Agent** - Rewrites the answer based on the critique and produces the final response.

Data flows from one stage to the next through shared session state via each agent's `output_key`,
which downstream agents read using `{state_key}` instruction templating.

> Runtime target: Agent Platform Colab Enterprise (Vertex AI authenticated).

## Step 1: Install dependencies

Versions are pinned for reproducibility. The built-in `google_search` tool ships with `google-adk`.

In [ ]:
%%writefile requirements.txt
ipykernel==7.3.0
jupyter==1.1.1
pandas==3.0.3
google-adk==1.18.0
litellm==1.83.9
google-cloud-aiplatform==1.148.1
requests==2.32.3

%pip install -q -r requirements.txt
print("Dependencies installed. If imports fail below, restart the runtime and continue.")

## Step 2: Configuration and authentication

Sets the GCP project, region, and model identifier, and routes all GenAI/ADK calls through Vertex AI. In Colab Enterprise the runtime service account authenticates automatically, so no key file is needed for Gemini.

In [ ]:
import os

import vertexai

# --- GCP / project settings ---
PROJECT_ID = "qwiklabs-gcp-02-b0d878248173"
LOCATION = "us-central1"

# --- Model identifier ---
GEMINI_MODEL = "gemini-2.5-flash"

# --- Route GenAI/ADK through Vertex AI ---
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Project: {PROJECT_ID}")
print(f"Gemini model: {GEMINI_MODEL} ({LOCATION})")

## Step 3: Define the four sub-agents

Each `LlmAgent` stores its result in shared session state via `output_key`. Downstream agents read
those values with `{state_key}` instruction templating:

- `search_agent` -> `initial_answer`
- `critique_agent` -> `critique` (reads `{initial_answer}`)
- `refine_agent` -> `final_answer` (reads `{initial_answer}` and `{critique}`)

The `search_agent` uses the built-in `google_search` tool. Because Gemini rejects mixing the
built-in search tool with any other tool, transfer is disabled on that agent so no auto-added
`transfer_to_agent` tool collides with it.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search

greeter_agent = Agent(
    name="greeter_agent",
    model=GEMINI_MODEL,
    description="Acknowledges the user's question and hands off to the workflow.",
    instruction=(
        "You are the greeter for the Cloud Security Advisor. In one friendly sentence, "
        "acknowledge the user's question and say that the team will research, critique, and "
        "refine an answer. Do NOT attempt to answer the question yourself."
    ),
    output_key="greeting",
)

search_agent = Agent(
    name="search_agent",
    model=GEMINI_MODEL,
    description="Researches and drafts the initial answer using Google Search.",
    instruction=(
        "You are a Google Cloud security researcher. Use google_search to research the user's "
        "question, then write a thorough, well-structured initial answer with concrete, "
        "actionable best practices."
    ),
    tools=[google_search],
    # google_search cannot be combined with an auto-added transfer tool (Gemini 400 error).
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
    output_key="initial_answer",
)

critique_agent = Agent(
    name="critique_agent",
    model=GEMINI_MODEL,
    description="Reviews the initial answer and recommends improvements.",
    instruction=(
        "You are a senior cloud security reviewer. Review the initial answer below for accuracy, "
        "completeness, clarity, and missing security considerations. Provide specific, actionable "
        "improvement recommendations as a bulleted list. Do not rewrite the answer.\n\n"
        "INITIAL ANSWER:\n{initial_answer}"
    ),
    output_key="critique",
)

refine_agent = Agent(
    name="refine_agent",
    model=GEMINI_MODEL,
    description="Rewrites the answer based on the critique.",
    instruction=(
        "Rewrite the answer to incorporate the critique. Produce ONLY the final, improved, "
        "well-structured response.\n\n"
        "ORIGINAL ANSWER:\n{initial_answer}\n\n"
        "CRITIQUE:\n{critique}"
    ),
    output_key="final_answer",
)

print("Defined greeter_agent, search_agent, critique_agent, refine_agent.")

## Step 4: Assemble the SequentialAgent

`SequentialAgent` runs its sub-agents in the exact order provided, sharing one session state across
all stages. This guarantees the strict Greeter -> Search -> Critique -> Refine pipeline.

In [ ]:
from google.adk.agents.sequential_agent import SequentialAgent

cloud_security_advisor = SequentialAgent(
    name="cloud_security_advisor",
    description="Greets, researches, critiques, and refines a security answer in strict order.",
    sub_agents=[greeter_agent, search_agent, critique_agent, refine_agent],
)

root_agent = cloud_security_advisor
print(f"Built SequentialAgent '{root_agent.name}' with stages:")
for index, sub_agent in enumerate(root_agent.sub_agents, start=1):
    print(f"  {index}. {sub_agent.name}")

## Step 5: Runner and event helper

We host the workflow in an `AdkApp` and run it with `stream_query`. The `run_workflow` helper prints
each streamed event labeled by its `author` (the acting sub-agent), so it is clear when the Greeter,
Search, Critique, and Refine agents are invoked. It also records the last text from each stage so
the answer's evolution can be displayed afterwards.

In [ ]:
from typing import Any, Dict

from vertexai.preview.reasoning_engines import AdkApp
from IPython.display import Markdown, display


def make_app(agent: Any) -> AdkApp:
    """Wrap an ADK agent in an AdkApp for running queries."""
    return AdkApp(agent=agent, enable_tracing=False)


def run_workflow(app: AdkApp, message: str, user_id: str = "student") -> Dict[str, str]:
    """Run the workflow, print events per sub-agent, and return the last text from each stage."""
    session = app.create_session(user_id=user_id)
    stage_text: Dict[str, str] = {}

    print(f"QUESTION: {message}\n" + "=" * 80)
    for event in app.stream_query(user_id=user_id, session_id=session.id, message=message):
        if not isinstance(event, dict):
            continue
        author = event.get("author", "unknown")
        for part in (event.get("content") or {}).get("parts", []) or []:
            if not isinstance(part, dict):
                continue
            text = part.get("text")
            if text:
                stage_text[author] = text
                print(f"\n----- [{author}] -----\n{text}")

    return stage_text


app = make_app(root_agent)
print("Runner ready. Events will print in stage order: greeter -> search -> critique -> refine.")

## Step 6: Run the workflow

This is the required demonstration. The streamed events show each sub-agent being invoked in order,
and the per-stage summary below shows how the response evolves from the initial draft, through the
critique, to the refined final answer.

In [ ]:
QUESTION = "What are the best practices for securing a private GKE cluster handling sensitive data?"

stages = run_workflow(app, QUESTION)

assert stages.get("refine_agent", "").strip(), "No final answer produced by refine_agent"

print("\n\n" + "=" * 80)
print("RESPONSE EVOLUTION THROUGH THE WORKFLOW")
print("=" * 80)
stage_titles = {
    "greeter_agent": "1. Greeter (acknowledgement)",
    "search_agent": "2. Search (initial answer)",
    "critique_agent": "3. Critique (recommended improvements)",
    "refine_agent": "4. Refine (final answer)",
}
for name, title in stage_titles.items():
    if name in stages:
        display(Markdown(f"### {title}\n{stages[name]}"))

## Expected output signals

In the event log above, look for:

- A short acknowledgement authored by `greeter_agent`.
- An initial, research-backed answer authored by `search_agent` (using `google_search`).
- A bulleted list of improvement recommendations authored by `critique_agent`.
- A rewritten, improved final answer authored by `refine_agent`.

Seeing the four authors appear in order is the evidence that the SequentialAgent workflow ran each
sub-agent and that the response was verified and refined before being returned.